> **Publication clean copy.** Outputs and execution counts were stripped from the evidence notebook (`../evidence/MCMEA_v2_regime_aware_governance.ipynb`, SHA-256 `163ba4d05d2e744e…`); code is unchanged. Run top-to-bottom with *Restart & Run All*. Exact recorded values are checked by `scripts/reproduce_recorded_outputs.py` (see `mcmea/README.md`).


# MCMEA v2: Regime-Aware Governance Evolution Under Signal and Noise

## What is MCMEA?

**MCMEA** stands for **Markov-Chain Macrostate Evolutionary Algorithm**. It is a governance simulation framework that combines two layers:

1. **Evolutionary search** over candidate governance configurations in an **S-Z space**
   - **S** = state capacity
   - **Z** = institutional constraint

2. **Markov macrostate analysis** over regime trajectories
   - to estimate persistence
   - switching
   - attractor basins
   - and long-run regime stability

In plain terms, MCMEA does not treat governance as a fixed ranking or a moral ladder. It treats governance as a **dynamic equilibrium landscape** in which configurations compete, adapt, persist, or fail under changing structural pressures.

---

## What is the governance idea behind this notebook?

The broader research idea is that governance should not be modeled as a single universally best regime. Instead, governance is better understood as a **structured space of competing basins**.

This notebook studies three governance topologies:

- **A_Strong_S**: strong state capacity, lower constraint
- **B_Strong_SZ**: high capacity + high constraint
- **C_Strong_Z**: strong institutional constraint, lower capacity

The purpose is not to prove one regime is morally superior. The purpose is to study:

- when one regime dominates,
- when multiple regimes genuinely compete,
- when shocks re-rank them,
- and when the system enters a **degeneracy regime** in which apparent winners are not clearly distinguishable.

---

## What does this notebook intend to show?

This notebook is designed to test a stronger version of MCMEA:

1. **Governance regimes should remain separate basins**
   - not be smoothed into one blended attractor

2. **Selection should be regime-aware**
   - the algorithm should first ask whether regime differences are statistically resolvable

3. **Noise matters structurally**
   - if signal is small relative to noise, regime winners may be unstable or misleading

4. **Search should adapt to distinguishability**
   - when regime resolution is low, the algorithm should explore more and exploit less

---

## Why revise the earlier MCMEA?

The earlier version used a `smooth_max(...)` aggregation over regime peaks. That made the landscape too smooth and effectively blended competing governance regimes into one softened surface. As a result, the simulation tended to collapse into a single basin too easily.

This revision incorporates the insight from the **Signal-Noise Regime** paper:

> Search outcomes should not be trusted unless the system can actually distinguish among the competing structures.

So MCMEA v2 adds:

- **explicit regime competition** instead of smoothed blending
- **regime-level noisy observation**
- **resolution diagnostics**
  - top-two regime gap
  - effective noise
  - regime resolution ratio \(R\)
- **adaptive search behavior**
  - low resolution -> more exploration
  - high resolution -> more exploitation

---

## Main research claim of this notebook

This notebook does **not** try to prove that one governance regime is always best.

It tries to show that:

> The apparent superiority of a governance basin depends on the structure of the search space, the shock context, and whether regime differences remain distinguishable under noise.

That is the main reason MCMEA was revised.



In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter



## Helper functions

These utilities handle clamping, entropy, Kendall tau, Gaussian peaks, and simple regime bookkeeping.


In [ ]:

def clamp(x, lo=0.0, hi=1.0):
    return max(lo, min(hi, x))


def shannon_entropy(probs):
    probs = np.asarray(probs, dtype=float)
    probs = probs[probs > 0]
    if len(probs) == 0:
        return 0.0
    return float(-np.sum(probs * np.log(probs)))


def kendall_tau_from_ranks(r1, r2):
    n = len(r1)
    if n < 2:
        return 0.0

    concordant = 0
    discordant = 0
    for i in range(n):
        for j in range(i + 1, n):
            a = r1[i] - r1[j]
            b = r2[i] - r2[j]
            if a * b > 0:
                concordant += 1
            elif a * b < 0:
                discordant += 1

    denom = concordant + discordant
    return float((concordant - discordant) / denom) if denom > 0 else 0.0


def gaussian_peak(S, Z, muS, muZ, wS=0.18, wZ=0.18):
    return np.exp(-((S - muS) ** 2 / (2 * wS ** 2) + (Z - muZ) ** 2 / (2 * wZ ** 2)))


def softmax(x, temp=1.0):
    x = np.asarray(x, dtype=float)
    x = x / max(temp, 1e-8)
    x = x - np.max(x)
    ex = np.exp(x)
    return ex / np.sum(ex)



## State representation

We optionally decompose **Z** into:
- rule of law
- democratic accountability

Then combine them into a single institutional-constraint axis for plotting and macrostate classification.


In [ ]:

USE_Z_COMPONENTS = True
W_ROL = 0.70
W_DA = 0.30


def compose_Z(rol, da):
    return clamp(W_ROL * rol + W_DA * da, 0.0, 1.0)



## Context and shocks

Here, context affects both:
- regime payoffs
- measurement noise

This makes shocks structurally meaningful rather than purely decorative.


In [ ]:

def sample_beta_context(rng):
    return {
        "threat": rng.uniform(0, 1),
        "volatility": rng.uniform(0, 1),
        "polarization": rng.uniform(0, 1),
    }


def beta_scalar(ctx):
    return (
        0.10 * (ctx["threat"] - 0.5)
        - 0.08 * (ctx["volatility"] - 0.5)
        - 0.06 * (ctx["polarization"] - 0.5)
    )


def sigma_ctx(ctx, sigma0=0.10, w_vol=0.18, w_pol=0.16, w_thr=0.06):
    return float(
        sigma0
        + w_vol * ctx["volatility"]
        + w_pol * ctx["polarization"]
        + w_thr * ctx["threat"]
    )



## Governance topology scores

This version does **not** smooth away regime competition.

Instead, it keeps **A**, **B**, and **C** as separate governance basins whose ranking can change with context.


In [ ]:

def topology_scores(S, Z, ctx):
    threat = ctx["threat"]
    volatility = ctx["volatility"]
    polarization = ctx["polarization"]

    # More balanced amplitudes than the earlier version
    peak_A = 1.25 * gaussian_peak(S, Z, muS=0.82, muZ=0.38, wS=0.16, wZ=0.18)
    peak_B = 1.25 * gaussian_peak(S, Z, muS=0.78, muZ=0.78, wS=0.16, wZ=0.16)
    peak_C = 1.25 * gaussian_peak(S, Z, muS=0.42, muZ=0.84, wS=0.18, wZ=0.16)

    # Context-specific structural favorability
    ctx_A = (
        0.45 * threat
        + 0.25 * volatility
        - 0.10 * polarization
    )
    ctx_B = (
        0.25 * (1 - volatility)
        + 0.25 * (1 - polarization)
        + 0.10 * (1 - abs(threat - 0.5))
    )
    ctx_C = (
        0.40 * (1 - threat)
        + 0.15 * (1 - volatility)
        - 0.10 * polarization
    )

    b = beta_scalar(ctx)

    Q_A = peak_A + ctx_A + b
    Q_B = peak_B + ctx_B + b
    Q_C = peak_C + ctx_C + b

    return {
        "A_Strong_S": float(Q_A),
        "B_Strong_SZ": float(Q_B),
        "C_Strong_Z": float(Q_C),
    }


def governance_latent_components(S, Z, ctx):
    return topology_scores(S, Z, ctx)



## Noisy observation and regime resolution

This is where the signal-noise paper enters directly.

For each candidate governance point, we evaluate the three regime scores under noise and compute:
- latent scores
- observed mean
- observed std
- robust score = mean - lambda * std

Then we estimate whether the top two regimes are distinguishable.


In [ ]:

def observe_regime_scores(S, Z, ctx, rng, m=5, lam=1.0):
    comps = governance_latent_components(S, Z, ctx)
    sig = sigma_ctx(ctx)

    obs = {}
    for name, q in comps.items():
        vals = q + rng.normal(0.0, sig, size=m)
        mu = float(np.mean(vals))
        sd = float(np.std(vals, ddof=0))
        robust = float(mu - lam * sd)
        obs[name] = {
            "latent": q,
            "mean": mu,
            "std": sd,
            "robust": robust,
        }
    return obs, sig


def regime_resolution(obs, sigma_eff):
    latents = sorted(
        [(k, v["latent"]) for k, v in obs.items()],
        key=lambda x: x[1],
        reverse=True
    )
    if len(latents) < 2:
        return {
            "R": 0.0,
            "delta": 0.0,
            "top1": None,
            "top2": None,
        }

    top1, q1 = latents[0]
    top2, q2 = latents[1]
    delta = float(q1 - q2)
    R = float(delta / sigma_eff) if sigma_eff > 0 else 0.0

    return {
        "R": R,
        "delta": delta,
        "top1": top1,
        "top2": top2,
    }


def select_regime_from_obs(obs, mode="robust", rng=None, temp=1.0):
    keys = list(obs.keys())

    if mode == "latent_max":
        return max(keys, key=lambda k: obs[k]["latent"])

    if mode == "robust_max":
        return max(keys, key=lambda k: obs[k]["robust"])

    if mode == "stochastic_robust":
        scores = np.array([obs[k]["robust"] for k in keys], dtype=float)
        probs = softmax(scores, temp=temp)
        idx = rng.choice(len(keys), p=probs)
        return keys[idx]

    raise ValueError(f"Unknown mode: {mode}")



## Macrostate classification

This is used for the Markov layer.


In [ ]:

def classify_point(S, Z):
    if S >= 0.68 and Z <= 0.58:
        return "A_Strong_S"
    if S >= 0.65 and Z >= 0.65:
        return "B_Strong_SZ"
    if Z >= 0.68 and S <= 0.58:
        return "C_Strong_Z"
    return "Hybrid"



## Evolutionary mechanics

The search becomes **adaptive**:
- low regime resolution -> more mutation, less elitism
- high regime resolution -> more exploitation


In [ ]:

def init_ind(rng):
    if USE_Z_COMPONENTS:
        return {"S": rng.random(), "rol": rng.random(), "da": rng.random()}
    return {"S": rng.random(), "Z": rng.random()}


def get_SZ(ind):
    if USE_Z_COMPONENTS:
        return ind["S"], compose_Z(ind["rol"], ind["da"])
    return ind["S"], ind["Z"]


def mutate(ind, rng, p=0.75, sigma=0.09):
    child = ind.copy()
    if rng.random() > p:
        return child

    if USE_Z_COMPONENTS:
        child["S"] = clamp(child["S"] + rng.normal(0, sigma))
        child["rol"] = clamp(child["rol"] + rng.normal(0, sigma))
        child["da"] = clamp(child["da"] + rng.normal(0, sigma))
    else:
        child["S"] = clamp(child["S"] + rng.normal(0, sigma))
        child["Z"] = clamp(child["Z"] + rng.normal(0, sigma))
    return child


def crossover(a, b, rng, alpha=0.45):
    def mix(x, y):
        t = rng.uniform(-alpha, 1 + alpha)
        return clamp((1 - t) * x + t * y)

    c1, c2 = a.copy(), b.copy()

    if USE_Z_COMPONENTS:
        for k in ["S", "rol", "da"]:
            c1[k] = mix(a[k], b[k])
            c2[k] = mix(b[k], a[k])
    else:
        for k in ["S", "Z"]:
            c1[k] = mix(a[k], b[k])
            c2[k] = mix(b[k], a[k])

    return c1, c2


def tournament(pop, fits, rng, k=3):
    idxs = rng.choice(len(pop), size=min(k, len(pop)), replace=False)
    best = max(idxs, key=lambda i: fits[i])
    return pop[best]


def diversity_grid(pop, tol=0.05):
    pts = []
    for ind in pop:
        S, Z = get_SZ(ind)
        pts.append((int(S / tol), int(Z / tol)))
    return len(set(pts)) / max(1, len(pop))


def adaptive_params(R_regime):
    if R_regime < 0.5:
        return {"elite": 4, "cx_rate": 0.40, "mut_rate": 0.90}
    if R_regime < 1.0:
        return {"elite": 6, "cx_rate": 0.55, "mut_rate": 0.80}
    return {"elite": 10, "cx_rate": 0.70, "mut_rate": 0.65}



## MCMEA v2 runner

This version preserves:
- separate regime basins
- regime-level noisy observation
- resolution diagnostics
- adaptive search behavior


In [ ]:

def run_mcmea_v2(
    pop_size=90,
    generations=60,
    base_elite=10,
    base_cx_rate=0.65,
    base_mut_rate=0.75,
    tour_k=3,
    m_eval=5,
    lambda_robust=1.0,
    seed=7,
    regime_select_mode="robust_max",
    regime_select_temp=1.0,
):
    rng = np.random.default_rng(seed)
    pop = [init_ind(rng) for _ in range(pop_size)]

    history = []
    macro_states = []
    ctxs = []

    diag = {
        "best_fitness": [],
        "best_regime_R": [],
        "best_regime_delta": [],
        "best_regime_top1": [],
        "best_regime_top2": [],
        "best_obs_std": [],
        "sigma_eff": [],
        "tau_latent_vs_observed": [],
        "pop_diversity": [],
        "elite_diversity": [],
        "adaptive_elite": [],
        "adaptive_cx_rate": [],
        "adaptive_mut_rate": [],
    }

    for g in range(generations):
        ctx = sample_beta_context(rng)
        ctxs.append(ctx)

        fit = np.empty(len(pop), dtype=float)
        best_obs_std_each = np.empty(len(pop), dtype=float)
        regime_R_each = np.empty(len(pop), dtype=float)
        regime_delta_each = np.empty(len(pop), dtype=float)
        latent_proxy = np.empty(len(pop), dtype=float)
        chosen_regime_each = []
        top1_each = []
        top2_each = []

        for i, ind in enumerate(pop):
            S, Z = get_SZ(ind)
            obs, sig = observe_regime_scores(S, Z, ctx, rng, m=m_eval, lam=lambda_robust)
            sigma_eff = sig / np.sqrt(max(1, m_eval))
            res = regime_resolution(obs, sigma_eff)

            chosen_regime = select_regime_from_obs(
                obs,
                mode=regime_select_mode,
                rng=rng,
                temp=regime_select_temp
            )

            fit[i] = obs[chosen_regime]["robust"]
            latent_proxy[i] = obs[chosen_regime]["latent"]
            best_obs_std_each[i] = obs[chosen_regime]["std"]
            regime_R_each[i] = res["R"]
            regime_delta_each[i] = res["delta"]
            chosen_regime_each.append(chosen_regime)
            top1_each.append(res["top1"])
            top2_each.append(res["top2"])

        order_obs = np.argsort(-fit, kind="mergesort")
        order_lat = np.argsort(-latent_proxy, kind="mergesort")

        pop_sorted = [pop[i] for i in order_obs]
        chosen_regime_sorted = [chosen_regime_each[i] for i in order_obs]

        snap = []
        for idx, ind in zip(order_obs, pop_sorted):
            S, Z = get_SZ(ind)
            snap.append({
                "S": float(S),
                "Z": float(Z),
                "fitness": float(fit[idx]),
                "latent_proxy": float(latent_proxy[idx]),
                "obs_std": float(best_obs_std_each[idx]),
                "regime_R": float(regime_R_each[idx]),
                "regime_delta": float(regime_delta_each[idx]),
                "chosen_regime": chosen_regime_each[idx],
                "top1_regime": top1_each[idx],
                "top2_regime": top2_each[idx],
                "macro": classify_point(S, Z),
            })
        history.append(snap)

        current_R = float(np.mean(regime_R_each))
        adapt = adaptive_params(current_R)

        elite_now = adapt["elite"]
        cx_now = adapt["cx_rate"]
        mut_now = adapt["mut_rate"]

        elites = pop_sorted[:elite_now]
        S_bar = float(np.mean([get_SZ(e)[0] for e in elites]))
        Z_bar = float(np.mean([get_SZ(e)[1] for e in elites]))
        macro_states.append(classify_point(S_bar, Z_bar))

        r_lat = np.empty(len(pop), dtype=int)
        r_obs = np.empty(len(pop), dtype=int)
        r_lat[order_lat] = np.arange(len(pop))
        r_obs[order_obs] = np.arange(len(pop))
        tau = kendall_tau_from_ranks(r_lat, r_obs)

        diag["best_fitness"].append(float(np.max(fit)))
        diag["best_regime_R"].append(float(regime_R_each[order_obs[0]]))
        diag["best_regime_delta"].append(float(regime_delta_each[order_obs[0]]))
        diag["best_regime_top1"].append(top1_each[order_obs[0]])
        diag["best_regime_top2"].append(top2_each[order_obs[0]])
        diag["best_obs_std"].append(float(best_obs_std_each[order_obs[0]]))
        diag["sigma_eff"].append(float(sig / np.sqrt(max(1, m_eval))))
        diag["tau_latent_vs_observed"].append(float(tau))
        diag["pop_diversity"].append(diversity_grid(pop, tol=0.05))
        diag["elite_diversity"].append(diversity_grid(elites, tol=0.05))
        diag["adaptive_elite"].append(int(elite_now))
        diag["adaptive_cx_rate"].append(float(cx_now))
        diag["adaptive_mut_rate"].append(float(mut_now))

        survivors = elites.copy()
        children = []

        while len(survivors) + len(children) < pop_size:
            p1 = tournament(pop, fit, rng, k=tour_k)
            p2 = tournament(pop, fit, rng, k=tour_k)

            if rng.random() < cx_now:
                c1, c2 = crossover(p1, p2, rng)
            else:
                c1, c2 = p1.copy(), p2.copy()

            c1 = mutate(c1, rng, p=mut_now)
            c2 = mutate(c2, rng, p=mut_now)

            children.append(c1)
            if len(survivors) + len(children) < pop_size:
                children.append(c2)

        pop = survivors + children

    return history, macro_states, ctxs, diag



## Markov utilities and winner entropy


In [ ]:

def build_markov(macro_states, state_list):
    idx = {s: i for i, s in enumerate(state_list)}
    n = len(state_list)
    C = np.zeros((n, n), dtype=float)

    for t in range(len(macro_states) - 1):
        i = idx[macro_states[t]]
        j = idx[macro_states[t + 1]]
        C[i, j] += 1

    P = C.copy()
    for i in range(n):
        rs = P[i].sum()
        if rs > 0:
            P[i] /= rs
        else:
            P[i, i] = 1.0

    return P, C


def elite_winner_bin(history, tol=0.05):
    last = history[-1]
    best = max(last, key=lambda x: x["fitness"])
    return (int(best["S"] / tol), int(best["Z"] / tol))


def run_many_v2(seeds, **kwargs):
    winners = []
    for sd in seeds:
        history, _, _, _ = run_mcmea_v2(seed=sd, **kwargs)
        winners.append(elite_winner_bin(history, tol=0.05))

    uniq, counts = np.unique(winners, axis=0, return_counts=True)
    probs = counts / counts.sum()

    H = shannon_entropy(probs)
    eff = float(np.exp(H))
    return {"H": H, "eff": eff, "uniq": uniq, "counts": counts}



## Example run

The default run uses:
- robust regime selection
- 5 noisy evaluations per candidate
- adaptive search behavior


In [ ]:

state_list = ["A_Strong_S", "B_Strong_SZ", "C_Strong_Z", "Hybrid"]

history, macro, ctxs, diag = run_mcmea_v2(
    pop_size=90,
    generations=60,
    m_eval=5,
    lambda_robust=1.0,
    seed=7,
    regime_select_mode="robust_max",
    regime_select_temp=1.0,
)



## Plots


In [ ]:

first = history[0]
last = history[-1]

plt.figure(figsize=(9, 6))
plt.scatter([p["S"] for p in first], [p["Z"] for p in first], s=18, alpha=0.20, label="first")
plt.scatter([p["S"] for p in last], [p["Z"] for p in last], s=24, alpha=0.85, label="final")
plt.xlabel("S (State Capacity)")
plt.ylabel("Z (Institutional Constraint)")
plt.title("MCMEA v2: S-Z Space")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()


In [ ]:

plt.figure(figsize=(10, 4))
plt.plot(diag["best_fitness"], label="best fitness")
plt.xlabel("Generation")
plt.ylabel("Score")
plt.title("Best Fitness Over Time")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()


In [ ]:

plt.figure(figsize=(10, 4))
plt.plot(diag["best_regime_R"], label="best regime R")
plt.plot(diag["tau_latent_vs_observed"], label="Kendall tau")
plt.xlabel("Generation")
plt.ylabel("Value")
plt.title("Resolution and Rank Stability")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()


In [ ]:

plt.figure(figsize=(10, 4))
plt.plot(diag["best_obs_std"], label="winner observed std")
plt.plot(diag["sigma_eff"], label="sigma_eff")
plt.xlabel("Generation")
plt.ylabel("Value")
plt.title("Observed Instability of Winner")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()


In [ ]:

plt.figure(figsize=(10, 4))
plt.plot(diag["pop_diversity"], label="population diversity")
plt.plot(diag["elite_diversity"], label="elite diversity")
plt.xlabel("Generation")
plt.ylabel("Grid diversity")
plt.title("Diversity Over Time")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()


In [ ]:

plt.figure(figsize=(10, 4))
plt.plot(diag["adaptive_elite"], label="adaptive elite")
plt.plot(diag["adaptive_cx_rate"], label="adaptive crossover rate")
plt.plot(diag["adaptive_mut_rate"], label="adaptive mutation rate")
plt.xlabel("Generation")
plt.ylabel("Value")
plt.title("Adaptive Search Settings")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()


In [ ]:

plt.figure(figsize=(10, 3.2))
plt.plot([state_list.index(s) for s in macro], marker="o", markersize=3, linewidth=1.2)
plt.yticks(range(len(state_list)), state_list)
plt.xlabel("Generation")
plt.ylabel("Dominant macrostate")
plt.title("Macro-state Trajectory")
plt.grid(True, alpha=0.2)
plt.show()


In [ ]:

P, C = build_markov(macro, state_list)

plt.figure(figsize=(7, 6))
plt.imshow(P, aspect="auto")
plt.xticks(range(len(state_list)), state_list, rotation=30, ha="right")
plt.yticks(range(len(state_list)), state_list)
plt.title("Estimated Markov Transition Matrix")
plt.colorbar()
plt.tight_layout()
plt.show()

print("State order:", state_list)
print("\nTransition counts C:\n", C.astype(int))
print("\nTransition probabilities P:\n", np.round(P, 3))



## Winner entropy across runs

This checks whether final winners remain concentrated or disperse across repeated runs.


In [ ]:

seeds = list(range(20, 60))
res = run_many_v2(
    seeds,
    pop_size=90,
    generations=50,
    m_eval=5,
    lambda_robust=1.0,
    regime_select_mode="robust_max",
    regime_select_temp=1.0,
)

print(f"Winner entropy H = {res['H']:.3f}")
print(f"Effective winners exp(H) = {res['eff']:.2f}")
print(f"Unique winner bins = {len(res['counts'])}")



## Summary table


In [ ]:

summary_df = pd.DataFrame({
    "generation": np.arange(len(diag["best_fitness"])),
    "best_fitness": diag["best_fitness"],
    "best_regime_R": diag["best_regime_R"],
    "best_regime_delta": diag["best_regime_delta"],
    "best_regime_top1": diag["best_regime_top1"],
    "best_regime_top2": diag["best_regime_top2"],
    "best_obs_std": diag["best_obs_std"],
    "sigma_eff": diag["sigma_eff"],
    "tau_latent_vs_observed": diag["tau_latent_vs_observed"],
    "pop_diversity": diag["pop_diversity"],
    "elite_diversity": diag["elite_diversity"],
    "adaptive_elite": diag["adaptive_elite"],
    "adaptive_cx_rate": diag["adaptive_cx_rate"],
    "adaptive_mut_rate": diag["adaptive_mut_rate"],
})
summary_df.head(10)
